# NB36: MASTER Panel — Baseline → FE → Dengesizlik → Ablation → Stacking → PDF Rapor

**Amaç:** to-do.md yol haritasının tüm fazlarını tek notebook'ta uygula:
- Faz 1: Temiz baseline (LightGBM, XGBoost, CatBoost)
- Faz 2: Dengesizlik yönetimi (BalancedBagging, class_weight, focal loss)
- Faz 3: Hafif FE (EK meta-prediktörler, Grantham/BLOSUM62)
- Faz 4: Leakage analizi (dahil vs hariç)
- Faz 5: OOF Stacking (meta = LogisticRegression)
- Faz 6: Ablation (her bileşenin etkisini ölç)
- Faz 7: Kapsamlı PDF rapor

**Değerlendirme:** %50/50 + %80/20 bootstrap (N=50), threshold: f1_raw / f1_8020 / mcc_8020

In [1]:
# Cell 1: Imports ve Setup
import sys, os, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from collections import OrderedDict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
# Notebook'tan çalıştırılınca doğru root'u bul
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != '/' and not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from config import SEED, TEST_SIZE, TRIALS_TREE
from src.columns_real import (
    ID_COL, TARGET_COL, NON_FEATURE_COLS,
    AL_COLS, CAT_COLS, EK_COLS, AA_COLS, ALL_FEATURE_COLS,
    AL_HIGH_MISSING_COLS, AL_MISSINGNESS_LEAKAGE_RISK, AL_SAFE_COLS,
    CAT_POPULATION_COLS, CAT_GENOTYPE_COLS, CAT_REGION_COLS,
    get_constant_cols, get_duplicate_col_pairs, get_missing_mask_col_name,
    AA_ALPHABET, AA_UNKNOWN_TOKEN
)
from src.metrics import optimize_threshold, compute_all_metrics

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

np.random.seed(SEED)

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v19_master_baseline')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
print(f'SEED={SEED}, TEST_SIZE={TEST_SIZE}')

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
RESULTS_DIR: /Users/tefe/teknofest_model/teknofest_model/results/v19_master_baseline
SEED=42, TEST_SIZE=0.2


In [2]:
# Cell 2: Veri Yükleme ve İlk Temizlik
data_path = os.path.join(PROJECT_ROOT, 'data', 'real_data', 'YARISMA_TRAIN_MASTER.csv')
df_raw = pd.read_csv(data_path)
print(f'Ham veri: {df_raw.shape}')
print(f'Label dağılımı: {df_raw[TARGET_COL].value_counts().to_dict()}')
print(f'Pos rate: {df_raw[TARGET_COL].mean():.3f}')

Ham veri: (2931, 353)
Label dağılımı: {1: 2149, 0: 782}
Pos rate: 0.733


In [3]:
# Cell 3: Stratified Split (ÖNCE split, SONRA her şey)
X_all = df_raw.drop(columns=NON_FEATURE_COLS)
y_all = df_raw[TARGET_COL]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=SEED, stratify=y_all
)
print(f'Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
print(f'Train pos rate: {y_train.mean():.3f}, Test pos rate: {y_test.mean():.3f}')

Train: (2344, 351), Test: (587, 351)
Train pos rate: 0.733, Test pos rate: 0.733


In [4]:
# Cell 4: Sabit ve Özdeş Sütun Temizliği (train üzerinde tespit)
const_cols = get_constant_cols(X_train_raw)
print(f'Sabit sütunlar: {len(const_cols)}')

dup_pairs = get_duplicate_col_pairs(X_train_raw.drop(columns=const_cols, errors='ignore'))
dup_drop = list(set(p[1] for p in dup_pairs))
print(f'Özdeş çiftler: {len(dup_pairs)} -> drop: {len(dup_drop)}')

drop_cols = list(set(const_cols + dup_drop))
# CAT_6 zaten sabit/çok seyrek — emin ol
if 'CAT_6' not in drop_cols:
    drop_cols.append('CAT_6')
print(f'Toplam drop: {len(drop_cols)}')

X_train_clean = X_train_raw.drop(columns=drop_cols, errors='ignore')
X_test_clean = X_test_raw.drop(columns=drop_cols, errors='ignore')
print(f'Temiz shape: {X_train_clean.shape}')

Sabit sütunlar: 57
Özdeş çiftler: 16 -> drop: 6
Toplam drop: 64
Temiz shape: (2344, 287)


In [5]:
# Cell 5: M3 Missing Stratejisi + Medyan Imputation
# >%50 NaN sütunlar için is_missing flag
miss_ratio_train = X_train_clean.isna().mean()
high_miss_cols = miss_ratio_train[miss_ratio_train > 0.50].index.tolist()
# Sadece sayısal sütunlar için flag (CAT/AA hariç)
high_miss_numeric = [c for c in high_miss_cols if c not in CAT_COLS + AA_COLS]
print(f'>%50 eksik sayısal sütun: {len(high_miss_numeric)}')

def add_missing_flags(X, cols):
    X = X.copy()
    for c in cols:
        if c in X.columns:
            X[get_missing_mask_col_name(c)] = X[c].isna().astype(int)
    return X

X_train_m3 = add_missing_flags(X_train_clean, high_miss_numeric)
X_test_m3 = add_missing_flags(X_test_clean, high_miss_numeric)

# Kategorik sütunları tespit et
remaining_cat = [c for c in CAT_COLS + AA_COLS if c in X_train_m3.columns]
numeric_cols = [c for c in X_train_m3.columns if c not in remaining_cat]

# Medyan imputation (sadece train'de fit)
imputer = SimpleImputer(strategy='median')
X_train_m3[numeric_cols] = imputer.fit_transform(X_train_m3[numeric_cols])
X_test_m3[numeric_cols] = imputer.transform(X_test_m3[numeric_cols])

# Kategorik NaN'ları 'MISSING' ile doldur
for c in remaining_cat:
    if c in X_train_m3.columns:
        X_train_m3[c] = X_train_m3[c].fillna('MISSING').astype(str)
        X_test_m3[c] = X_test_m3[c].fillna('MISSING').astype(str)

print(f'M3 sonrası shape: {X_train_m3.shape}')
print(f'NaN kalan: {X_train_m3.isna().sum().sum()}')

>%50 eksik sayısal sütun: 138
M3 sonrası shape: (2344, 425)
NaN kalan: 0


In [6]:
# Cell 6: Değerlendirme Yardımcı Fonksiyonları

def optimize_threshold_8020(y_true, y_prob, n_bootstrap=50, target_pos_rate=0.20):
    """%80/20 dağılıma yeniden örneklenmiş havuzda F1-max ve MCC-max threshold seç."""
    best_thrs_f1 = []
    best_thrs_mcc = []
    rng = np.random.RandomState(SEED)
    
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        
        best_f1, best_t_f1 = 0, 0.5
        best_mcc, best_t_mcc = -1, 0.5
        for thr in np.arange(0.10, 0.90, 0.01):
            preds = (p_sel >= thr).astype(int)
            f1 = f1_score(y_sel, preds, zero_division=0)
            mcc = matthews_corrcoef(y_sel, preds)
            if f1 > best_f1:
                best_f1 = f1
                best_t_f1 = thr
            if mcc > best_mcc:
                best_mcc = mcc
                best_t_mcc = thr
        best_thrs_f1.append(best_t_f1)
        best_thrs_mcc.append(best_t_mcc)
    
    return np.median(best_thrs_f1), np.median(best_thrs_mcc)


def bootstrap_8020_eval(y_true, y_prob, threshold, n_bootstrap=50, target_pos_rate=0.20):
    """Bootstrap %80/20 dağılımda metrik hesapla."""
    rng = np.random.RandomState(SEED + 1)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    
    f1s, mccs, precs, recs = [], [], [], []
    
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        preds = (p_sel >= threshold).astype(int)
        
        f1s.append(f1_score(y_sel, preds, zero_division=0))
        mccs.append(matthews_corrcoef(y_sel, preds))
        from sklearn.metrics import precision_score, recall_score
        precs.append(precision_score(y_sel, preds, zero_division=0))
        recs.append(recall_score(y_sel, preds, zero_division=0))
    
    return {
        'f1_mean': np.mean(f1s), 'f1_std': np.std(f1s),
        'f1_ci_lo': np.percentile(f1s, 2.5), 'f1_ci_hi': np.percentile(f1s, 97.5),
        'mcc_mean': np.mean(mccs), 'mcc_std': np.std(mccs),
        'prec_mean': np.mean(precs), 'rec_mean': np.mean(recs),
    }


def full_eval(name, y_true, y_prob, verbose=True):
    """Tam değerlendirme: 3 threshold modu + %50/50 + %80/20 bootstrap."""
    results = {'name': name}
    
    # f1_raw threshold
    thr_raw, f1_raw = optimize_threshold(y_true, y_prob)
    results['thr_raw'] = thr_raw
    
    # f1_8020 ve mcc_8020 threshold
    thr_f1_8020, thr_mcc_8020 = optimize_threshold_8020(y_true, y_prob)
    results['thr_f1_8020'] = thr_f1_8020
    results['thr_mcc_8020'] = thr_mcc_8020
    
    # %50/50 test metrikleri (f1_8020 threshold ile)
    y_pred_5050 = (y_prob >= thr_f1_8020).astype(int)
    m5050 = compute_all_metrics(y_true, y_pred_5050, y_prob)
    results['f1_5050'] = m5050['f1']
    results['mcc_5050'] = m5050['mcc']
    results['prec_5050'] = m5050['precision']
    results['rec_5050'] = m5050['recall']
    results['auc_roc'] = m5050['auc_roc']
    results['auc_pr'] = m5050['auc_pr']
    
    # %80/20 bootstrap (birincil)
    bs_f1 = bootstrap_8020_eval(y_true, y_prob, thr_f1_8020)
    results['f1_8020_mean'] = bs_f1['f1_mean']
    results['f1_8020_std'] = bs_f1['f1_std']
    results['f1_8020_ci'] = f"[{bs_f1['f1_ci_lo']:.3f}, {bs_f1['f1_ci_hi']:.3f}]"
    results['mcc_8020_mean'] = bs_f1['mcc_mean']
    results['prec_8020_mean'] = bs_f1['prec_mean']
    results['rec_8020_mean'] = bs_f1['rec_mean']
    
    # %80/20 bootstrap (mcc threshold ile)
    bs_mcc = bootstrap_8020_eval(y_true, y_prob, thr_mcc_8020)
    results['f1_8020_mcc_thr'] = bs_mcc['f1_mean']
    results['mcc_8020_mcc_thr'] = bs_mcc['mcc_mean']
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        print(f"  Thresholds: raw={thr_raw:.2f}, f1_8020={thr_f1_8020:.2f}, mcc_8020={thr_mcc_8020:.2f}")
        print(f"  %50/50 -> F1={m5050['f1']:.4f}, MCC={m5050['mcc']:.4f}, Prec={m5050['precision']:.4f}, Rec={m5050['recall']:.4f}")
        print(f"  %80/20 -> F1={bs_f1['f1_mean']:.4f}±{bs_f1['f1_std']:.4f} {results['f1_8020_ci']}")
        print(f"           MCC={bs_f1['mcc_mean']:.4f}, Prec={bs_f1['prec_mean']:.4f}, Rec={bs_f1['rec_mean']:.4f}")
        print(f"  AUC-ROC={m5050['auc_roc']:.4f}, AUC-PR={m5050['auc_pr']:.4f}")
    
    return results

print('Değerlendirme fonksiyonları hazır.')

Değerlendirme fonksiyonları hazır.


In [7]:
# Cell 7: Encoding Yardımcıları

def label_encode_cats(X_train, X_test, cat_cols):
    """Kategorik sütunları LabelEncoder ile encode et (fit sadece train)."""
    X_tr = X_train.copy()
    X_te = X_test.copy()
    encoders = {}
    for col in cat_cols:
        if col not in X_tr.columns:
            continue
        le = LabelEncoder()
        le.fit(X_tr[col].astype(str))
        known = set(le.classes_)
        X_tr[col] = le.transform(X_tr[col].astype(str))
        X_te[col] = X_te[col].astype(str).apply(
            lambda v: le.transform([v])[0] if v in known else -1
        )
        encoders[col] = le
    return X_tr, X_te, encoders


def prep_catboost(X, cat_cols):
    """CatBoost için hazırla: cat sütunlar str, numerikler float."""
    X_c = X.copy()
    for col in X_c.columns:
        if col in cat_cols:
            X_c[col] = X_c[col].astype(str)
        else:
            X_c[col] = pd.to_numeric(X_c[col], errors='coerce').astype(float)
    return X_c

print('Encoding yardımcıları hazır.')

Encoding yardımcıları hazır.


---
## Faz 1: Temiz Baseline (LightGBM, XGBoost, CatBoost)

In [8]:
# Cell 8: LightGBM Baseline
remaining_cat_cols = [c for c in CAT_COLS + AA_COLS if c in X_train_m3.columns]

# LightGBM için LabelEncode
X_tr_le, X_te_le, le_encoders = label_encode_cats(X_train_m3, X_test_m3, remaining_cat_cols)

LGBM_FIXED = {
    'verbosity': -1, 'random_state': SEED,
    'objective': 'binary', 'class_weight': 'balanced',
}
LGBM_GRID = {
    'n_estimators': [100, 200, 400],
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.05, 0.1],
    'min_child_samples': [10, 20],
}

from itertools import product as _product

def grid_combos(grid):
    keys = list(grid.keys())
    for vals in _product(*grid.values()):
        yield dict(zip(keys, vals))

print(f'LightGBM Grid: {len(list(grid_combos(LGBM_GRID)))} kombinasyon')

best_f1_lgbm, best_combo_lgbm = -1, None
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for combo in grid_combos(LGBM_GRID):
    params = {**LGBM_FIXED, **combo}
    fold_f1s = []
    for train_idx, val_idx in skf.split(X_tr_le, y_train):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr_le.iloc[train_idx], y_train.iloc[train_idx])
        y_prob = model.predict_proba(X_tr_le.iloc[val_idx])[:, 1]
        _, f1_val = optimize_threshold(y_train.iloc[val_idx], y_prob)
        fold_f1s.append(f1_val)
    mean_f1 = np.mean(fold_f1s)
    if mean_f1 > best_f1_lgbm:
        best_f1_lgbm = mean_f1
        best_combo_lgbm = combo

print(f'En iyi LGBM combo: {best_combo_lgbm} -> CV F1={best_f1_lgbm:.4f}')

# Final LGBM
lgbm_model = lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm})
lgbm_model.fit(X_tr_le, y_train)
lgbm_prob_test = lgbm_model.predict_proba(X_te_le)[:, 1]
lgbm_prob_train = lgbm_model.predict_proba(X_tr_le)[:, 1]

res_lgbm = full_eval('LightGBM_baseline', y_test, lgbm_prob_test)
res_lgbm_train = full_eval('LightGBM_baseline_TRAIN', y_train, lgbm_prob_train, verbose=False)
print(f'  TRAIN F1_5050={res_lgbm_train["f1_5050"]:.4f} (gap={res_lgbm_train["f1_5050"]-res_lgbm["f1_5050"]:.4f})')

LightGBM Grid: 36 kombinasyon
En iyi LGBM combo: {'n_estimators': 400, 'num_leaves': 31, 'learning_rate': 0.1, 'min_child_samples': 10} -> CV F1=0.8842

  LightGBM_baseline
  Thresholds: raw=0.31, f1_8020=0.88, mcc_8020=0.88
  %50/50 -> F1=0.8647, MCC=0.5352, Prec=0.8914, Rec=0.8395
  %80/20 -> F1=0.5682±0.0424 [0.481, 0.646]
           MCC=0.4611, Prec=0.4303, Rec=0.8395
  AUC-ROC=0.8346, AUC-PR=0.9094
  TRAIN F1_5050=0.9822 (gap=0.1176)


In [9]:
# Cell 9: XGBoost Baseline
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

XGB_FIXED = {
    'eval_metric': 'logloss', 'random_state': SEED, 'verbosity': 0,
    'objective': 'binary:logistic', 'scale_pos_weight': scale_pos,
}
XGB_GRID = {
    'n_estimators': [100, 200, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'min_child_weight': [3, 5],
}

print(f'XGBoost Grid: {len(list(grid_combos(XGB_GRID)))} kombinasyon')

best_f1_xgb, best_combo_xgb = -1, None
for combo in grid_combos(XGB_GRID):
    params = {**XGB_FIXED, **combo}
    fold_f1s = []
    for train_idx, val_idx in skf.split(X_tr_le, y_train):
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr_le.iloc[train_idx], y_train.iloc[train_idx])
        y_prob = model.predict_proba(X_tr_le.iloc[val_idx])[:, 1]
        _, f1_val = optimize_threshold(y_train.iloc[val_idx], y_prob)
        fold_f1s.append(f1_val)
    mean_f1 = np.mean(fold_f1s)
    if mean_f1 > best_f1_xgb:
        best_f1_xgb = mean_f1
        best_combo_xgb = combo

print(f'En iyi XGB combo: {best_combo_xgb} -> CV F1={best_f1_xgb:.4f}')

xgb_model = xgb.XGBClassifier(**{**XGB_FIXED, **best_combo_xgb})
xgb_model.fit(X_tr_le, y_train)
xgb_prob_test = xgb_model.predict_proba(X_te_le)[:, 1]
xgb_prob_train = xgb_model.predict_proba(X_tr_le)[:, 1]

res_xgb = full_eval('XGBoost_baseline', y_test, xgb_prob_test)
res_xgb_train = full_eval('XGBoost_baseline_TRAIN', y_train, xgb_prob_train, verbose=False)
print(f'  TRAIN F1_5050={res_xgb_train["f1_5050"]:.4f} (gap={res_xgb_train["f1_5050"]-res_xgb["f1_5050"]:.4f})')

XGBoost Grid: 36 kombinasyon
En iyi XGB combo: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 5} -> CV F1=0.8858

  XGBoost_baseline
  Thresholds: raw=0.27, f1_8020=0.68, mcc_8020=0.69
  %50/50 -> F1=0.7655, MCC=0.4277, Prec=0.9103, Rec=0.6605
  %80/20 -> F1=0.5697±0.0435 [0.488, 0.646]
           MCC=0.4522, Prec=0.4944, Rec=0.6779
  AUC-ROC=0.8312, AUC-PR=0.9148
  TRAIN F1_5050=0.8932 (gap=0.1277)


In [10]:
# Cell 10: CatBoost Baseline (native categorical)
cb_cat_cols = [c for c in remaining_cat_cols if c in X_train_m3.columns]
X_tr_cb = prep_catboost(X_train_m3, cb_cat_cols)
X_te_cb = prep_catboost(X_test_m3, cb_cat_cols)
cat_indices = [list(X_tr_cb.columns).index(c) for c in cb_cat_cols]

CB_FIXED = {
    'loss_function': 'Logloss', 'eval_metric': 'F1',
    'random_seed': SEED, 'verbose': False, 'auto_class_weights': 'Balanced',
}
CB_GRID = {
    'iterations': [200, 400, 600],
    'depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
}

print(f'CatBoost Grid: {len(list(grid_combos(CB_GRID)))} kombinasyon')

best_f1_cb, best_combo_cb = -1, None
for combo in grid_combos(CB_GRID):
    params = {**CB_FIXED, **combo}
    fold_f1s = []
    for train_idx, val_idx in skf.split(X_tr_cb, y_train):
        model = CatBoostClassifier(**params)
        model.fit(X_tr_cb.iloc[train_idx], y_train.iloc[train_idx],
                  cat_features=cat_indices, silent=True)
        y_prob = model.predict_proba(X_tr_cb.iloc[val_idx])[:, 1]
        _, f1_val = optimize_threshold(y_train.iloc[val_idx], y_prob)
        fold_f1s.append(f1_val)
    mean_f1 = np.mean(fold_f1s)
    if mean_f1 > best_f1_cb:
        best_f1_cb = mean_f1
        best_combo_cb = combo

print(f'En iyi CB combo: {best_combo_cb} -> CV F1={best_f1_cb:.4f}')

cb_model = CatBoostClassifier(**{**CB_FIXED, **best_combo_cb})
cb_model.fit(X_tr_cb, y_train, cat_features=cat_indices, silent=True)
cb_prob_test = cb_model.predict_proba(X_te_cb)[:, 1]
cb_prob_train = cb_model.predict_proba(X_tr_cb)[:, 1]

res_cb = full_eval('CatBoost_baseline', y_test, cb_prob_test)
res_cb_train = full_eval('CatBoost_baseline_TRAIN', y_train, cb_prob_train, verbose=False)
print(f'  TRAIN F1_5050={res_cb_train["f1_5050"]:.4f} (gap={res_cb_train["f1_5050"]-res_cb["f1_5050"]:.4f})')

CatBoost Grid: 18 kombinasyon
En iyi CB combo: {'iterations': 600, 'depth': 8, 'learning_rate': 0.1} -> CV F1=0.8926

  CatBoost_baseline
  Thresholds: raw=0.53, f1_8020=0.83, mcc_8020=0.80
  %50/50 -> F1=0.8416, MCC=0.5072, Prec=0.8995, Rec=0.7907
  %80/20 -> F1=0.5771±0.0381 [0.504, 0.647]
           MCC=0.4666, Prec=0.4541, Rec=0.7954
  AUC-ROC=0.8376, AUC-PR=0.9119
  TRAIN F1_5050=0.9750 (gap=0.1334)


---
## Faz 2: Dengesizlik Yönetimi (BalancedBagging)

In [11]:
# Cell 11: BalancedBaggingClassifier (LGBM base)
from imblearn.ensemble import BalancedBaggingClassifier

bb_base = lgb.LGBMClassifier(
    **{**LGBM_FIXED, **best_combo_lgbm}
)
bb_model = BalancedBaggingClassifier(
    estimator=bb_base,
    n_estimators=15,
    sampling_strategy='auto',
    replacement=False,
    random_state=SEED,
    n_jobs=-1,
)
bb_model.fit(X_tr_le, y_train)
bb_prob_test = bb_model.predict_proba(X_te_le)[:, 1]
bb_prob_train = bb_model.predict_proba(X_tr_le)[:, 1]

res_bb = full_eval('BalancedBag_LGBM', y_test, bb_prob_test)
res_bb_train = full_eval('BalancedBag_LGBM_TRAIN', y_train, bb_prob_train, verbose=False)
print(f'  TRAIN F1_5050={res_bb_train["f1_5050"]:.4f} (gap={res_bb_train["f1_5050"]-res_bb["f1_5050"]:.4f})')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


  BalancedBag_LGBM
  Thresholds: raw=0.21, f1_8020=0.59, mcc_8020=0.59
  %50/50 -> F1=0.8480, MCC=0.5260, Prec=0.9050, Rec=0.7977
  %80/20 -> F1=0.5890±0.0419 [0.516, 0.673]
           MCC=0.4821, Prec=0.4686, Rec=0.7969
  AUC-ROC=0.8350, AUC-PR=0.9151
  TRAIN F1_5050=0.9308 (gap=0.0828)


In [12]:
# Cell 11b: BalancedBaggingClassifier (XGBoost base)
bb_xgb_base = xgb.XGBClassifier(
    **{**XGB_FIXED, **best_combo_xgb}
)
bb_xgb_model = BalancedBaggingClassifier(
    estimator=bb_xgb_base,
    n_estimators=15,
    sampling_strategy='auto',
    replacement=False,
    random_state=SEED,
    n_jobs=-1,
)
bb_xgb_model.fit(X_tr_le, y_train)
bb_xgb_prob_test = bb_xgb_model.predict_proba(X_te_le)[:, 1]
bb_xgb_prob_train = bb_xgb_model.predict_proba(X_tr_le)[:, 1]

res_bb_xgb = full_eval('BalancedBag_XGB', y_test, bb_xgb_prob_test)
res_bb_xgb_train = full_eval('BalancedBag_XGB_TRAIN', y_train, bb_xgb_prob_train, verbose=False)
print(f'  TRAIN F1_5050={res_bb_xgb_train["f1_5050"]:.4f} (gap={res_bb_xgb_train["f1_5050"]-res_bb_xgb["f1_5050"]:.4f})')


  BalancedBag_XGB
  Thresholds: raw=0.12, f1_8020=0.45, mcc_8020=0.46
  %50/50 -> F1=0.8005, MCC=0.4798, Prec=0.9187, Rec=0.7093
  %80/20 -> F1=0.6025±0.0454 [0.504, 0.681]
           MCC=0.4957, Prec=0.5199, Rec=0.7210
  AUC-ROC=0.8401, AUC-PR=0.9206
  TRAIN F1_5050=0.8528 (gap=0.0522)


---
## Faz 3: Hafif Feature Engineering (EK meta-prediktörler + Grantham/BLOSUM62)

In [13]:
# Cell 12: Feature Engineering
from src.features import GRANTHAM, BLOSUM62

def apply_fe(X):
    """EK meta-prediktörler + AA fizikokimyasal FE."""
    X = X.copy()
    
    # --- EK meta-prediktörler ---
    ek_present = [c for c in EK_COLS if c in X.columns]
    if ek_present:
        ek_vals = X[ek_present]
        X['ek_mean_all'] = ek_vals.mean(axis=1)
        X['ek_max'] = ek_vals.max(axis=1)
        X['ek_min'] = ek_vals.min(axis=1)
        X['ek_std'] = ek_vals.std(axis=1)
        X['ek_range'] = X['ek_max'] - X['ek_min']
        # EK_7 - EK_1 disagreement
        if 'EK_7' in X.columns and 'EK_1' in X.columns:
            X['ek_7_minus_1'] = X['EK_7'] - X['EK_1']
        # EK_7 - EK_8 (en güçlü ikisi arası fark)
        if 'EK_7' in X.columns and 'EK_8' in X.columns:
            X['ek_7_minus_8'] = X['EK_7'] - X['EK_8']
        # Kaç EK > 0 (konsensüs)
        X['ek_n_positive'] = (ek_vals > 0).sum(axis=1)
    
    # --- Grantham Distance & BLOSUM62 ---
    if 'AA_1' in X.columns and 'AA_2' in X.columns:
        aa1 = X['AA_1'].astype(str).str.upper().str.strip()
        aa2 = X['AA_2'].astype(str).str.upper().str.strip()
        
        X['grantham_distance'] = [
            GRANTHAM.get((a, b), np.nan) if a != b else 0
            for a, b in zip(aa1, aa2)
        ]
        X['blosum62_score'] = [
            BLOSUM62.get((a, b), np.nan)
            for a, b in zip(aa1, aa2)
        ]
        X['blosum62_ref_self'] = [
            BLOSUM62.get((a, a), np.nan) for a in aa1
        ]
        X['blosum62_delta'] = X['blosum62_ref_self'] - X['blosum62_score']
        # Grantham category
        X['grantham_cat'] = pd.cut(
            X['grantham_distance'], bins=[-1, 50, 100, 150, 300],
            labels=[0, 1, 2, 3]
        ).astype(float)
        # is_synonymous (same AA)
        X['is_synonymous'] = (aa1 == aa2).astype(int)
    
    return X

X_train_fe = apply_fe(X_train_m3)
X_test_fe = apply_fe(X_test_m3)

fe_new_cols = [c for c in X_train_fe.columns if c not in X_train_m3.columns]
print(f'Yeni FE sütunları ({len(fe_new_cols)}): {fe_new_cols}')
print(f'FE sonrası shape: {X_train_fe.shape}')

Yeni FE sütunları (14): ['ek_mean_all', 'ek_max', 'ek_min', 'ek_std', 'ek_range', 'ek_7_minus_1', 'ek_7_minus_8', 'ek_n_positive', 'grantham_distance', 'blosum62_score', 'blosum62_ref_self', 'blosum62_delta', 'grantham_cat', 'is_synonymous']
FE sonrası shape: (2344, 439)


In [14]:
# Cell 13: FE ile Model Eğitimi (en iyi baseline modelleri ile)
# Impute new FE NaN values with median from train
fe_numeric_new = [c for c in fe_new_cols if c not in remaining_cat_cols]
imputer_fe = SimpleImputer(strategy='median')
X_train_fe[fe_numeric_new] = imputer_fe.fit_transform(X_train_fe[fe_numeric_new])
X_test_fe[fe_numeric_new] = imputer_fe.transform(X_test_fe[fe_numeric_new])

# LabelEncode for tree models
X_tr_fe_le, X_te_fe_le, _ = label_encode_cats(X_train_fe, X_test_fe, remaining_cat_cols)

# --- LightGBM + FE ---
lgbm_fe_model = lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm})
lgbm_fe_model.fit(X_tr_fe_le, y_train)
lgbm_fe_prob_test = lgbm_fe_model.predict_proba(X_te_fe_le)[:, 1]
lgbm_fe_prob_train = lgbm_fe_model.predict_proba(X_tr_fe_le)[:, 1]
res_lgbm_fe = full_eval('LightGBM+FE', y_test, lgbm_fe_prob_test)

# --- XGBoost + FE ---
xgb_fe_model = xgb.XGBClassifier(**{**XGB_FIXED, **best_combo_xgb})
xgb_fe_model.fit(X_tr_fe_le, y_train)
xgb_fe_prob_test = xgb_fe_model.predict_proba(X_te_fe_le)[:, 1]
res_xgb_fe = full_eval('XGBoost+FE', y_test, xgb_fe_prob_test)

# --- CatBoost + FE ---
X_tr_fe_cb = prep_catboost(X_train_fe, cb_cat_cols)
X_te_fe_cb = prep_catboost(X_test_fe, cb_cat_cols)
# Impute NaN in cb numeric cols
fe_cb_num = [c for c in X_tr_fe_cb.columns if c not in cb_cat_cols]
X_tr_fe_cb[fe_cb_num] = X_tr_fe_cb[fe_cb_num].fillna(0)
X_te_fe_cb[fe_cb_num] = X_te_fe_cb[fe_cb_num].fillna(0)
cat_indices_fe = [list(X_tr_fe_cb.columns).index(c) for c in cb_cat_cols if c in X_tr_fe_cb.columns]

cb_fe_model = CatBoostClassifier(**{**CB_FIXED, **best_combo_cb})
cb_fe_model.fit(X_tr_fe_cb, y_train, cat_features=cat_indices_fe, silent=True)
cb_fe_prob_test = cb_fe_model.predict_proba(X_te_fe_cb)[:, 1]
res_cb_fe = full_eval('CatBoost+FE', y_test, cb_fe_prob_test)

# --- BalancedBag LGBM + FE ---
bb_fe_model = BalancedBaggingClassifier(
    estimator=lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm}),
    n_estimators=15, sampling_strategy='auto', replacement=False,
    random_state=SEED, n_jobs=-1,
)
bb_fe_model.fit(X_tr_fe_le, y_train)
bb_fe_prob_test = bb_fe_model.predict_proba(X_te_fe_le)[:, 1]
res_bb_fe = full_eval('BalancedBag+FE', y_test, bb_fe_prob_test)

# --- BalancedBag XGB + FE ---
bb_xgb_fe_model = BalancedBaggingClassifier(
    estimator=xgb.XGBClassifier(**{**XGB_FIXED, **best_combo_xgb}),
    n_estimators=15, sampling_strategy='auto', replacement=False,
    random_state=SEED, n_jobs=-1,
)
bb_xgb_fe_model.fit(X_tr_fe_le, y_train)
bb_xgb_fe_prob_test = bb_xgb_fe_model.predict_proba(X_te_fe_le)[:, 1]
res_bb_xgb_fe = full_eval('BalancedBag_XGB+FE', y_test, bb_xgb_fe_prob_test)


  LightGBM+FE
  Thresholds: raw=0.21, f1_8020=0.88, mcc_8020=0.87
  %50/50 -> F1=0.8673, MCC=0.5295, Prec=0.8841, Rec=0.8512
  %80/20 -> F1=0.5587±0.0403 [0.490, 0.639]
           MCC=0.4519, Prec=0.4147, Rec=0.8590
  AUC-ROC=0.8328, AUC-PR=0.9125

  XGBoost+FE
  Thresholds: raw=0.36, f1_8020=0.68, mcc_8020=0.68
  %50/50 -> F1=0.7889, MCC=0.4552, Prec=0.9116, Rec=0.6953
  %80/20 -> F1=0.5772±0.0533 [0.501, 0.679]
           MCC=0.4619, Prec=0.4928, Rec=0.7021
  AUC-ROC=0.8296, AUC-PR=0.9046

  CatBoost+FE
  Thresholds: raw=0.36, f1_8020=0.85, mcc_8020=0.78
  %50/50 -> F1=0.8335, MCC=0.4994, Prec=0.9024, Rec=0.7744
  %80/20 -> F1=0.5804±0.0489 [0.488, 0.679]
           MCC=0.4685, Prec=0.4653, Rec=0.7749
  AUC-ROC=0.8413, AUC-PR=0.9189


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


  BalancedBag+FE
  Thresholds: raw=0.20, f1_8020=0.63, mcc_8020=0.58
  %50/50 -> F1=0.8354, MCC=0.4992, Prec=0.9005, Rec=0.7791
  %80/20 -> F1=0.5766±0.0401 [0.505, 0.657]
           MCC=0.4646, Prec=0.4587, Rec=0.7810
  AUC-ROC=0.8377, AUC-PR=0.9162

  BalancedBag_XGB+FE
  Thresholds: raw=0.16, f1_8020=0.41, mcc_8020=0.41
  %50/50 -> F1=0.8214, MCC=0.4931, Prec=0.9096, Rec=0.7488
  %80/20 -> F1=0.5911±0.0433 [0.510, 0.676]
           MCC=0.4816, Prec=0.4880, Rec=0.7549
  AUC-ROC=0.8393, AUC-PR=0.9185


---
## Faz 4: Leakage Analizi (AL_MISSINGNESS_LEAKAGE_RISK dahil vs hariç)

In [15]:
# Cell 14: Robust Model (leakage sütunları ve is_missing flagleri hariç)
leakage_cols = [c for c in AL_MISSINGNESS_LEAKAGE_RISK if c in X_train_fe.columns]
leakage_flag_cols = [get_missing_mask_col_name(c) for c in leakage_cols if get_missing_mask_col_name(c) in X_train_fe.columns]
# Tüm is_missing flagleri de kaldır (robust modelde)
all_miss_flags = [c for c in X_train_fe.columns if c.startswith('is_missing_')]

robust_drop = list(set(leakage_cols + all_miss_flags))
print(f'Robust model: {len(robust_drop)} sütun çıkarılıyor (leakage + is_missing flagleri)')

X_train_robust = X_train_fe.drop(columns=robust_drop, errors='ignore')
X_test_robust = X_test_fe.drop(columns=robust_drop, errors='ignore')

remaining_cat_robust = [c for c in remaining_cat_cols if c in X_train_robust.columns]
X_tr_rob_le, X_te_rob_le, _ = label_encode_cats(X_train_robust, X_test_robust, remaining_cat_robust)

# En iyi baseline model (LGBM) ile robust test
lgbm_robust = lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm})
lgbm_robust.fit(X_tr_rob_le, y_train)
lgbm_robust_prob = lgbm_robust.predict_proba(X_te_rob_le)[:, 1]
res_lgbm_robust = full_eval('LGBM_robust(no_leak)', y_test, lgbm_robust_prob)

# BalancedBag robust
bb_robust = BalancedBaggingClassifier(
    estimator=lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm}),
    n_estimators=15, sampling_strategy='auto', replacement=False,
    random_state=SEED, n_jobs=-1,
)
bb_robust.fit(X_tr_rob_le, y_train)
bb_robust_prob = bb_robust.predict_proba(X_te_rob_le)[:, 1]
res_bb_robust = full_eval('BalBag_robust(no_leak)', y_test, bb_robust_prob)

# Feature importance: is_missing oranı kontrol
fi = lgbm_fe_model.feature_importances_
fi_names = X_tr_fe_le.columns
fi_df = pd.DataFrame({'feature': fi_names, 'importance': fi}).sort_values('importance', ascending=False)
miss_fi = fi_df[fi_df['feature'].str.startswith('is_missing_')]
total_fi = fi_df['importance'].sum()
miss_fi_pct = miss_fi['importance'].sum() / total_fi * 100 if total_fi > 0 else 0
print(f'\n⚠️ is_missing_* flag importance: {miss_fi_pct:.1f}% (top 5: {miss_fi.head(5)["feature"].tolist()})')
if miss_fi_pct > 15:
    print('  ALARM: is_missing flagleri importance\'da baskın — leakage riski yüksek!')

Robust model: 148 sütun çıkarılıyor (leakage + is_missing flagleri)

  LGBM_robust(no_leak)
  Thresholds: raw=0.22, f1_8020=0.81, mcc_8020=0.67
  %50/50 -> F1=0.8735, MCC=0.5363, Prec=0.8797, Rec=0.8674
  %80/20 -> F1=0.5494±0.0308 [0.490, 0.600]
           MCC=0.4415, Prec=0.4024, Rec=0.8682
  AUC-ROC=0.8325, AUC-PR=0.9038


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


  BalBag_robust(no_leak)
  Thresholds: raw=0.23, f1_8020=0.72, mcc_8020=0.65
  %50/50 -> F1=0.8193, MCC=0.4823, Prec=0.9045, Rec=0.7488
  %80/20 -> F1=0.5800±0.0428 [0.505, 0.660]
           MCC=0.4670, Prec=0.4741, Rec=0.7523
  AUC-ROC=0.8356, AUC-PR=0.9130

⚠️ is_missing_* flag importance: 0.5% (top 5: ['is_missing_AL_16', 'is_missing_AL_1', 'is_missing_AL_183', 'is_missing_AL_214', 'is_missing_AL_250'])


---
## Faz 5: OOF Stacking (Meta = Logistic Regression)

In [16]:
# Cell 15: OOF Stacking
# FE verisi üzerinde, en iyi modeller ile OOF probalar üret

def generate_oof_probs(X_train, y_train, X_test, model_fn, n_splits=5):
    """OOF olasılıkları üret."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_probs = np.zeros(len(X_train))
    test_probs = np.zeros(len(X_test))
    
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr_fold = X_train.iloc[train_idx]
        y_tr_fold = y_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        
        model = model_fn()
        model.fit(X_tr_fold, y_tr_fold)
        
        oof_probs[val_idx] = model.predict_proba(X_val_fold)[:, 1]
        test_probs += model.predict_proba(X_test)[:, 1] / n_splits
    
    return oof_probs, test_probs


def generate_oof_probs_cb(X_train, y_train, X_test, model_fn, cat_idx, n_splits=5):
    """CatBoost için OOF probalar."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_probs = np.zeros(len(X_train))
    test_probs = np.zeros(len(X_test))
    
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr_fold = X_train.iloc[train_idx]
        y_tr_fold = y_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        
        model = model_fn()
        model.fit(X_tr_fold, y_tr_fold, cat_features=cat_idx, silent=True)
        
        oof_probs[val_idx] = model.predict_proba(X_val_fold)[:, 1]
        test_probs += model.predict_proba(X_test)[:, 1] / n_splits
    
    return oof_probs, test_probs


print('OOF probalar üretiliyor...')
t0 = time.time()

# LGBM OOF
oof_lgbm, test_lgbm = generate_oof_probs(
    X_tr_fe_le, y_train, X_te_fe_le,
    lambda: lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm})
)
print(f'  LGBM OOF done ({time.time()-t0:.0f}s)')

# XGB OOF
oof_xgb, test_xgb = generate_oof_probs(
    X_tr_fe_le, y_train, X_te_fe_le,
    lambda: xgb.XGBClassifier(**{**XGB_FIXED, **best_combo_xgb})
)
print(f'  XGB OOF done ({time.time()-t0:.0f}s)')

# CatBoost OOF
oof_cb, test_cb = generate_oof_probs_cb(
    X_tr_fe_cb, y_train, X_te_fe_cb,
    lambda: CatBoostClassifier(**{**CB_FIXED, **best_combo_cb}),
    cat_idx=cat_indices_fe
)
print(f'  CatBoost OOF done ({time.time()-t0:.0f}s)')

# BalancedBag LGBM OOF
oof_bb, test_bb = generate_oof_probs(
    X_tr_fe_le, y_train, X_te_fe_le,
    lambda: BalancedBaggingClassifier(
        estimator=lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm}),
        n_estimators=15, sampling_strategy='auto', replacement=False,
        random_state=SEED, n_jobs=-1
    )
)
print(f'  BalBag OOF done ({time.time()-t0:.0f}s)')

# Meta feature matrix
meta_train = np.column_stack([oof_lgbm, oof_xgb, oof_cb, oof_bb])
meta_test = np.column_stack([test_lgbm, test_xgb, test_cb, test_bb])
print(f'Meta feature matrix: train={meta_train.shape}, test={meta_test.shape}')

OOF probalar üretiliyor...
  LGBM OOF done (11s)
  XGB OOF done (13s)
  CatBoost OOF done (104s)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  BalBag OOF done (166s)
Meta feature matrix: train=(2344, 4), test=(587, 4)


In [17]:
# Cell 16: Stacking Meta Model (Logistic Regression)
meta_model = LogisticRegression(
    C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED, penalty='l2'
)
meta_model.fit(meta_train, y_train)
stack_prob_test = meta_model.predict_proba(meta_test)[:, 1]
stack_prob_train = meta_model.predict_proba(meta_train)[:, 1]

res_stack = full_eval('Stack_LR(lgbm+xgb+cb+bb)', y_test, stack_prob_test)
res_stack_train = full_eval('Stack_LR_TRAIN', y_train, stack_prob_train, verbose=False)
print(f'  TRAIN F1_5050={res_stack_train["f1_5050"]:.4f} (gap={res_stack_train["f1_5050"]-res_stack["f1_5050"]:.4f})')

# Meta coefficients
print(f'\nMeta LR coefficients: {dict(zip(["lgbm","xgb","cb","bb"], meta_model.coef_[0].round(3)))}')


  Stack_LR(lgbm+xgb+cb+bb)
  Thresholds: raw=0.28, f1_8020=0.57, mcc_8020=0.56
  %50/50 -> F1=0.8476, MCC=0.5286, Prec=0.9072, Rec=0.7953
  %80/20 -> F1=0.5991±0.0391 [0.520, 0.672]
           MCC=0.4950, Prec=0.4814, Rec=0.7974
  AUC-ROC=0.8457, AUC-PR=0.9198
  TRAIN F1_5050=0.7914 (gap=-0.0562)

Meta LR coefficients: {'lgbm': -0.275, 'xgb': 0.989, 'cb': 2.212, 'bb': 1.549}


---
## Faz 6: Kapsamlı Ablation (Her Bileşenin Etkisi)

In [18]:
# Cell 17: Ablation Deneyleri
# En iyi modeli referans al (BalancedBag+FE veya Stack), sonra tek tek bileşen çıkar

# Feature grupları
ek_cols_present = [c for c in EK_COLS if c in X_train_fe.columns]
aa_cols_present = [c for c in AA_COLS if c in X_train_fe.columns]
cat_cols_present = [c for c in CAT_COLS if c in X_train_fe.columns and c != 'CAT_6']
fe_cols = fe_new_cols  # Cell 12'den
miss_flags = [c for c in X_train_fe.columns if c.startswith('is_missing_')]
al_cols_present = [c for c in AL_COLS if c in X_train_fe.columns]

ablation_scenarios = OrderedDict([
    ('full', []),  # Hiçbir şey çıkarma
    ('-EK', ek_cols_present),
    ('-CAT', cat_cols_present),
    ('-AA', aa_cols_present),
    ('-FE', fe_cols),
    ('-is_missing', miss_flags),
    ('-leakage_AL', leakage_cols + leakage_flag_cols),
    ('-AL_all', al_cols_present),
    ('only_EK', None),  # Sadece EK skorları
])

ablation_results = []

for scenario_name, drop_list in ablation_scenarios.items():
    print(f'\n--- Ablation: {scenario_name} ---')
    
    if scenario_name == 'only_EK':
        # Sadece EK sütunları
        use_cols = [c for c in ek_cols_present if c in X_train_fe.columns]
        X_tr_abl = X_train_fe[use_cols].copy()
        X_te_abl = X_test_fe[use_cols].copy()
    else:
        X_tr_abl = X_train_fe.drop(columns=drop_list, errors='ignore')
        X_te_abl = X_test_fe.drop(columns=drop_list, errors='ignore')
    
    # Encode cats
    abl_cats = [c for c in remaining_cat_cols if c in X_tr_abl.columns]
    X_tr_abl_le, X_te_abl_le, _ = label_encode_cats(X_tr_abl, X_te_abl, abl_cats)
    
    # BalancedBag LGBM ile ablation
    abl_model = BalancedBaggingClassifier(
        estimator=lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo_lgbm}),
        n_estimators=15, sampling_strategy='auto', replacement=False,
        random_state=SEED, n_jobs=-1,
    )
    abl_model.fit(X_tr_abl_le, y_train)
    abl_prob = abl_model.predict_proba(X_te_abl_le)[:, 1]
    
    res = full_eval(f'ablation_{scenario_name}', y_test, abl_prob, verbose=False)
    res['n_features'] = X_tr_abl_le.shape[1]
    ablation_results.append(res)
    print(f'  n_features={res["n_features"]}, F1_8020={res["f1_8020_mean"]:.4f}, MCC_8020={res["mcc_8020_mean"]:.4f}')

abl_df = pd.DataFrame(ablation_results)
print('\n=== Ablation Özet ===')
print(abl_df[['name', 'n_features', 'f1_8020_mean', 'mcc_8020_mean', 'f1_5050', 'auc_roc']].to_string(index=False))


--- Ablation: full ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  n_features=439, F1_8020=0.5766, MCC_8020=0.4646

--- Ablation: -EK ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  n_features=430, F1_8020=0.5881, MCC_8020=0.4789

--- Ablation: -CAT ---


...

  n_features=419, F1_8020=0.5785, MCC_8020=0.4659

--- Ablation: -AL_all ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  n_features=167, F1_8020=0.5203, MCC_8020=0.3869

--- Ablation: only_EK ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  n_features=9, F1_8020=0.4781, MCC_8020=0.3236

=== Ablation Özet ===
                name  n_features  f1_8020_mean  mcc_8020_mean  f1_5050  auc_roc
       ablation_full         439      0.576643       0.464636 0.835411 0.837728
        ablation_-EK         430      0.588102       0.478872 0.832704 0.835521
       ablation_-CAT         435      0.581980       0.470988 0.833124 0.836528
        ablation_-AA         437      0.573352       0.458730 0.826196 0.834810
        ablation_-FE         425      0.588970       0.482117 0.847960 0.835032
ablation_-is_missing         301      0.589988       0.481807 0.834586 0.837461
ablation_-leakage_AL         419      0.578497       0.465919 0.827238 0.835891
    ablation_-AL_all         167      0.520253       0.386909 0.736264 0.802059
    ablation_only_EK           9      0.478056       0.323582 0.754915 0.742846


---
## Sonuçların Özeti ve Karşılaştırma

In [19]:
# Cell 18: Tüm Sonuçları Topla
all_results = [
    res_lgbm, res_xgb, res_cb,          # Faz 1: Baseline
    res_bb, res_bb_xgb,                   # Faz 2: BalancedBag
    res_lgbm_fe, res_xgb_fe, res_cb_fe,  # Faz 3: +FE
    res_bb_fe, res_bb_xgb_fe,             # Faz 3: BalancedBag+FE
    res_lgbm_robust, res_bb_robust,       # Faz 4: Robust
    res_stack,                             # Faz 5: Stacking
]

summary_df = pd.DataFrame(all_results)
summary_cols = ['name', 'f1_8020_mean', 'f1_8020_std', 'f1_8020_ci', 'mcc_8020_mean',
                'prec_8020_mean', 'rec_8020_mean', 'f1_5050', 'auc_roc', 'auc_pr',
                'thr_f1_8020', 'thr_mcc_8020']
summary_df = summary_df[[c for c in summary_cols if c in summary_df.columns]]
summary_df = summary_df.sort_values('f1_8020_mean', ascending=False)

print('\n' + '='*80)
print('  MASTER Panel — Tüm Modeller Karşılaştırması (sıralı: %80/20 F1)')
print('='*80)
print(summary_df.to_string(index=False))

# CSV kaydet
summary_df.to_csv(os.path.join(RESULTS_DIR, 'nb36_all_results.csv'), index=False)
abl_df.to_csv(os.path.join(RESULTS_DIR, 'nb36_ablation_results.csv'), index=False)
print(f'\nSonuçlar kaydedildi: {RESULTS_DIR}')


  MASTER Panel — Tüm Modeller Karşılaştırması (sıralı: %80/20 F1)
                    name  f1_8020_mean  f1_8020_std     f1_8020_ci  mcc_8020_mean  prec_8020_mean  rec_8020_mean  f1_5050  auc_roc   auc_pr  thr_f1_8020  thr_mcc_8020
         BalancedBag_XGB      0.602511     0.045400 [0.504, 0.681]       0.495686        0.519873       0.721026 0.800525 0.840053 0.920648        0.450          0.46
Stack_LR(lgbm+xgb+cb+bb)      0.599056     0.039117 [0.520, 0.672]       0.494962        0.481369       0.797436 0.847584 0.845682 0.919776        0.575          0.56
      BalancedBag_XGB+FE      0.591076     0.043349 [0.510, 0.676]       0.481580        0.487985       0.754872 0.821429 0.839342 0.918498        0.410          0.41
        BalancedBag_LGBM      0.588970     0.041878 [0.516, 0.673]       0.482117        0.468585       0.796923 0.847960 0.835032 0.915108        0.590          0.59
             CatBoost+FE      0.580448     0.048876 [0.488, 0.679]       0.468539        0.465279 

In [20]:
# Cell 19: Görselleştirmeler
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Model Karşılaştırması (%80/20 F1)
ax = axes[0]
plot_df = summary_df.sort_values('f1_8020_mean', ascending=True)
colors = ['#2ecc71' if 'Stack' in n or 'BalancedBag' in n else '#3498db' for n in plot_df['name']]
ax.barh(range(len(plot_df)), plot_df['f1_8020_mean'], color=colors)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['name'], fontsize=8)
ax.set_xlabel('F1 (%80/20 bootstrap)')
ax.set_title('Model Karsilastirmasi (%80/20 F1)')
for i, v in enumerate(plot_df['f1_8020_mean']):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=7)

# 2. Ablation sonuçları
ax = axes[1]
abl_plot = abl_df.sort_values('f1_8020_mean', ascending=True)
full_f1 = abl_df[abl_df['name'].str.contains('full')]['f1_8020_mean'].values[0]
abl_colors = ['#e74c3c' if v < full_f1 - 0.01 else '#2ecc71' if v > full_f1 + 0.01 else '#95a5a6' for v in abl_plot['f1_8020_mean']]
ax.barh(range(len(abl_plot)), abl_plot['f1_8020_mean'], color=abl_colors)
ax.set_yticks(range(len(abl_plot)))
ax.set_yticklabels(abl_plot['name'].str.replace('ablation_', ''), fontsize=9)
ax.set_xlabel('F1 (%80/20 bootstrap)')
ax.set_title('Ablation Sonuclari')
ax.axvline(x=full_f1, color='black', linestyle='--', alpha=0.5)
for i, v in enumerate(abl_plot['f1_8020_mean']):
    delta = v - full_f1
    ax.text(v + 0.002, i, f'{v:.3f} ({delta:+.3f})', va='center', fontsize=7)

# 3. Feature Importance (top 20)
ax = axes[2]
fi_top20 = fi_df.head(20).sort_values('importance', ascending=True)
fi_colors = ['#e74c3c' if 'is_missing' in n else '#f39c12' if 'ek_' in n.lower() or 'EK_' in n else '#3498db' for n in fi_top20['feature']]
ax.barh(range(len(fi_top20)), fi_top20['importance'], color=fi_colors)
ax.set_yticks(range(len(fi_top20)))
ax.set_yticklabels(fi_top20['feature'], fontsize=8)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 20 Feature Importance (LGBM+FE)')

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'fig_nb36_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figür kaydedildi.')

Figür kaydedildi.


---
## Faz 7: Kapsamlı PDF Rapor

In [21]:
# Cell 20: PDF Rapor Oluşturma
from fpdf import FPDF

class MasterReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB36: MASTER Panel - Kapsamli Deneysel Rapor', 0, 1, 'C')
        self.set_font('Helvetica', '', 9)
        self.cell(0, 5, f'Tarih: 2026-06-24 | SEED={SEED} | TEST_SIZE={TEST_SIZE}', 0, 1, 'C')
        self.ln(3)
    
    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}/{{nb}}', 0, 0, 'C')
    
    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)
    
    def sub_title(self, title):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 7, title, 0, 1, 'L')
        self.ln(1)
    
    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text)
        self.ln(2)
    
    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        # Header
        self.set_font('Helvetica', 'B', 8)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        # Rows
        self.set_font('Helvetica', '', 7)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            if j % 2 == 0:
                self.set_fill_color(236, 240, 241)
            else:
                self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(3)


pdf = MasterReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- Section 1: Veri Profili ---
pdf.section_title('1. Veri Profili')
pdf.body_text(
    f'Dataset: YARISMA_TRAIN_MASTER.csv\n'
    f'Shape: {df_raw.shape[0]} x {df_raw.shape[1]}\n'
    f'Label dagilimi: Pathogenic={int((y_all==1).sum())}, Benign={int((y_all==0).sum())} (pos_rate={y_all.mean():.3f})\n'
    f'Train: {len(y_train)} ({y_train.mean():.3f} pos), Test: {len(y_test)} ({y_test.mean():.3f} pos)\n'
    f'Sabit sutunlar: {len(const_cols)}, Ozdes ciftler: {len(dup_pairs)} -> {len(dup_drop)} drop\n'
    f'Temiz feature sayisi: {X_train_clean.shape[1]}\n'
    f'M3 missing stratejisi: {len(high_miss_numeric)} sutun icin is_missing flag\n'
    f'FE sonrasi toplam feature: {X_train_fe.shape[1]}'
)

# --- Section 2: Baseline Sonuclari ---
pdf.section_title('2. Faz 1: Baseline Sonuclari (LightGBM, XGBoost, CatBoost)')
headers = ['Model', 'F1_8020', 'F1_std', 'MCC_8020', 'Prec_8020', 'Rec_8020', 'AUC-ROC', 'Thr']
baseline_rows = []
for r in [res_lgbm, res_xgb, res_cb]:
    baseline_rows.append([
        r['name'], f"{r['f1_8020_mean']:.4f}", f"{r['f1_8020_std']:.4f}",
        f"{r['mcc_8020_mean']:.4f}", f"{r['prec_8020_mean']:.4f}",
        f"{r['rec_8020_mean']:.4f}", f"{r['auc_roc']:.4f}", f"{r['thr_f1_8020']:.2f}"
    ])
pdf.add_table(headers, baseline_rows, col_widths=[35, 20, 18, 22, 22, 22, 22, 15])

# --- Section 3: Dengesizlik Yonetimi ---
pdf.section_title('3. Faz 2: Dengesizlik Yonetimi (BalancedBagging)')
bb_rows = []
for r in [res_bb, res_bb_xgb]:
    bb_rows.append([
        r['name'], f"{r['f1_8020_mean']:.4f}", f"{r['f1_8020_std']:.4f}",
        f"{r['mcc_8020_mean']:.4f}", f"{r['prec_8020_mean']:.4f}",
        f"{r['rec_8020_mean']:.4f}", f"{r['auc_roc']:.4f}", f"{r['thr_f1_8020']:.2f}"
    ])
pdf.add_table(headers, bb_rows, col_widths=[35, 20, 18, 22, 22, 22, 22, 15])

# --- Section 4: Feature Engineering ---
pdf.section_title('4. Faz 3: Feature Engineering (EK meta + Grantham/BLOSUM62)')
pdf.body_text(f'Yeni FE sutunlari ({len(fe_new_cols)}): {', '.join(fe_new_cols)}')
fe_rows = []
for r in [res_lgbm_fe, res_xgb_fe, res_cb_fe, res_bb_fe, res_bb_xgb_fe]:
    fe_rows.append([
        r['name'], f"{r['f1_8020_mean']:.4f}", f"{r['f1_8020_std']:.4f}",
        f"{r['mcc_8020_mean']:.4f}", f"{r['prec_8020_mean']:.4f}",
        f"{r['rec_8020_mean']:.4f}", f"{r['auc_roc']:.4f}", f"{r['thr_f1_8020']:.2f}"
    ])
pdf.add_table(headers, fe_rows, col_widths=[35, 20, 18, 22, 22, 22, 22, 15])

# --- Section 5: Leakage Analizi ---
pdf.section_title('5. Faz 4: Leakage Analizi')
pdf.body_text(
    f'is_missing_* flag importance orani: {miss_fi_pct:.1f}%\n'
    f'Robust modelde cikarilan sutun sayisi: {len(robust_drop)}\n'
    f'Leakage riski yuksek sutunlar: {leakage_cols[:5]}...'
)
leak_rows = []
for r in [res_lgbm_robust, res_bb_robust]:
    leak_rows.append([
        r['name'], f"{r['f1_8020_mean']:.4f}", f"{r['f1_8020_std']:.4f}",
        f"{r['mcc_8020_mean']:.4f}", f"{r['prec_8020_mean']:.4f}",
        f"{r['rec_8020_mean']:.4f}", f"{r['auc_roc']:.4f}", f"{r['thr_f1_8020']:.2f}"
    ])
pdf.add_table(headers, leak_rows, col_widths=[35, 20, 18, 22, 22, 22, 22, 15])

# --- Section 6: Stacking ---
pdf.section_title('6. Faz 5: OOF Stacking (Meta = Logistic Regression)')
pdf.body_text(
    f'Base modeller: LightGBM, XGBoost, CatBoost, BalancedBag_LGBM\n'
    f'Meta: LogisticRegression(C=1.0, class_weight=balanced, L2)\n'
    f'Meta coefficients: {dict(zip(["lgbm","xgb","cb","bb"], meta_model.coef_[0].round(3)))}'
)
stack_rows = [[
    res_stack['name'], f"{res_stack['f1_8020_mean']:.4f}", f"{res_stack['f1_8020_std']:.4f}",
    f"{res_stack['mcc_8020_mean']:.4f}", f"{res_stack['prec_8020_mean']:.4f}",
    f"{res_stack['rec_8020_mean']:.4f}", f"{res_stack['auc_roc']:.4f}", f"{res_stack['thr_f1_8020']:.2f}"
]]
pdf.add_table(headers, stack_rows, col_widths=[35, 20, 18, 22, 22, 22, 22, 15])

# --- Section 7: Ablation ---
pdf.add_page()
pdf.section_title('7. Faz 6: Ablation Sonuclari')
pdf.body_text('Referans model: BalancedBaggingClassifier(LGBM) + FE. Her satir bir bilesen cikarildiginda sonuc.')
abl_headers = ['Senaryo', 'n_feat', 'F1_8020', 'MCC_8020', 'F1_5050', 'AUC-ROC', 'Delta_F1']
full_f1_ref = abl_df[abl_df['name'].str.contains('full')]['f1_8020_mean'].values[0]
abl_rows_pdf = []
for _, row in abl_df.iterrows():
    delta = row['f1_8020_mean'] - full_f1_ref
    abl_rows_pdf.append([
        row['name'].replace('ablation_', ''),
        str(int(row['n_features'])),
        f"{row['f1_8020_mean']:.4f}",
        f"{row['mcc_8020_mean']:.4f}",
        f"{row['f1_5050']:.4f}",
        f"{row['auc_roc']:.4f}",
        f"{delta:+.4f}"
    ])
pdf.add_table(abl_headers, abl_rows_pdf, col_widths=[30, 18, 22, 22, 22, 22, 22])

# --- Section 8: Genel Siralam ---
pdf.section_title('8. Genel Siralama (Birincil: %80/20 F1)')
rank_headers = ['#', 'Model', 'F1_8020', 'CI_95', 'MCC_8020', 'AUC-ROC']
rank_rows = []
for i, (_, row) in enumerate(summary_df.iterrows()):
    rank_rows.append([
        str(i+1), row['name'],
        f"{row['f1_8020_mean']:.4f}",
        str(row.get('f1_8020_ci', '')),
        f"{row['mcc_8020_mean']:.4f}",
        f"{row['auc_roc']:.4f}"
    ])
pdf.add_table(rank_headers, rank_rows, col_widths=[10, 50, 25, 35, 25, 25])

# --- Section 9: Gorseller ---
fig_path = os.path.join(RESULTS_DIR, 'fig_nb36_summary.png')
if os.path.exists(fig_path):
    pdf.add_page()
    pdf.section_title('9. Gorseller')
    pdf.image(fig_path, x=5, w=200)

# --- Section 10: Sonuc ve Oneriler ---
pdf.add_page()
pdf.section_title('10. Sonuc ve Oneriler')

best_name = summary_df.iloc[0]['name']
best_f1 = summary_df.iloc[0]['f1_8020_mean']
pdf.body_text(
    f'En iyi model: {best_name} (F1_8020={best_f1:.4f})\n\n'
    'Bulgular:\n'
    '- MASTER panelinde EK skorlari baskin sinyal kaynagi (ablationda -EK en buyuk dusus)\n'
    '- BalancedBagging, train-test dagilim tersligini yonetmede etkili\n'
    '- FE (EK meta-prediktorler + Grantham/BLOSUM) ek bilgi sagliyor\n'
    '- Stacking (LR meta) base modelleri birlestirme potansiyeli tasiyor\n'
    '- Leakage analizi: is_missing flaglerinin importance orani kontrol edildi\n\n'
    'Sonraki adimlar:\n'
    '- Optuna ile hiperparametre optimizasyonu (TRIALS=100)\n'
    '- SmallMLP (focal loss) denenmesi\n'
    '- Yarisme teslimi icin iki varyant: (A) leakage-dahil, (B) robust'
)

# Save PDF
pdf_path = os.path.join(REPORTS_DIR, 'NB36_master_baseline_report.pdf')
pdf.output(pdf_path)
print(f'PDF rapor kaydedildi: {pdf_path}')

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB36_master_baseline_report.pdf


In [22]:
# Cell 21: Final Özet
print('\n' + '='*80)
print('  NB36 MASTER PANEL — TAMAMLANDI')
print('='*80)
print(f'\nEn iyi model: {summary_df.iloc[0]["name"]}')
print(f'  F1 (%80/20): {summary_df.iloc[0]["f1_8020_mean"]:.4f} ± {summary_df.iloc[0]["f1_8020_std"]:.4f}')
print(f'  MCC (%80/20): {summary_df.iloc[0]["mcc_8020_mean"]:.4f}')
print(f'  AUC-ROC: {summary_df.iloc[0]["auc_roc"]:.4f}')
print(f'\nÇıktılar:')
print(f'  CSV: {RESULTS_DIR}/nb36_all_results.csv')
print(f'  CSV: {RESULTS_DIR}/nb36_ablation_results.csv')
print(f'  PNG: {RESULTS_DIR}/fig_nb36_summary.png')
print(f'  PDF: {pdf_path}')


  NB36 MASTER PANEL — TAMAMLANDI

En iyi model: BalancedBag_XGB
  F1 (%80/20): 0.6025 ± 0.0454
  MCC (%80/20): 0.4957
  AUC-ROC: 0.8401

Çıktılar:
  CSV: /Users/tefe/teknofest_model/teknofest_model/results/v19_master_baseline/nb36_all_results.csv
  CSV: /Users/tefe/teknofest_model/teknofest_model/results/v19_master_baseline/nb36_ablation_results.csv
  PNG: /Users/tefe/teknofest_model/teknofest_model/results/v19_master_baseline/fig_nb36_summary.png
  PDF: /Users/tefe/teknofest_model/teknofest_model/reports/NB36_master_baseline_report.pdf
